# 01 · Exploración de los datos (EDA)

**Proyecto 22 · Detector de noticias falsas** — Marco Andrés Rico

Pasos 5 y 6 del formato estándar (criterio 3 de la rúbrica). Objetivo: conocer el Spanish Fake News Corpus,
**cuantificar** sus problemas de calidad y los riesgos de fuga **antes** de modelar.

> **Regla de uso de las pruebas en este notebook.** El análisis de contenido (clases, temas, fuentes, longitud,
> estilo) se hace **solo con la partición de entrenamiento**. De las particiones de prueba (DD y FD) solo se
> revisan estructura, formato, faltantes y duplicados contra entrenamiento, que son controles de integridad
> necesarios para evitar fuga. Ninguna decisión de modelado se toma mirando etiquetas de prueba.

In [1]:
import os, sys
from pathlib import Path
# Ejecutar desde la raíz del repositorio, con src/ en la ruta de importación
RAIZ = Path.cwd() if (Path.cwd() / "configs").exists() else Path.cwd().parent
os.chdir(RAIZ); sys.path.insert(0, str(RAIZ / "src"))
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 160)
from fakenews import config, data
from fakenews.plots import estilo, guardar, AZUL, NARANJA, AQUA, TEXTO_2
estilo()
print("Raíz del proyecto:", RAIZ)

Raíz del proyecto: /home/claude/detector-noticias-falsas


## 1. Acceso y verificación de integridad
Se descargan los archivos desde el commit fijado y se comprueba su SHA‑256 (evidencia de acceso).

In [2]:
rutas = data.descargar()
crudos = {k: pd.read_excel(v) for k, v in rutas.items()}
for k, d in crudos.items():
    print(f"{k:13s} {d.shape[0]:4d} filas · columnas: {list(d.columns)}")

  OK train.xlsx (748,241 bytes, sha256 b091a8fa842e…)
  OK development.xlsx (362,324 bytes, sha256 5a8941521702…)
  OK test.xlsx (923,248 bytes, sha256 11e38355804f…)


entrenamiento  676 filas · columnas: ['Id', 'Category', 'Topic', 'Source', 'Headline', 'Text', 'Link']
prueba_dd      295 filas · columnas: ['Id', 'Category', 'Topic', 'Source', 'Headline', 'Text', 'Link']
prueba_fd      572 filas · columnas: ['ID', 'CATEGORY', 'TOPICS', 'SOURCE', 'HEADLINE', 'TEXT', 'LINK']


## 2. Esquema y faltantes
La v2 usa nombres en mayúsculas y una etiqueta booleana; se armoniza al esquema de v1.

In [3]:
faltantes = pd.DataFrame({k: d.rename(columns=str.lower).rename(columns={"topics": "topic"}).isna().sum()
                          for k, d in crudos.items()})
faltantes

,entrenamiento,prueba_dd,prueba_fd
id,0,0,0
category,0,0,0
topic,0,0,0
source,0,0,7
headline,0,0,72
text,0,0,0
link,0,0,3


In [4]:
print("Tipos de la etiqueta:", {k: str(d.iloc[:, 1].dtype) for k, d in crudos.items()})
print("Valores v2:", crudos["prueba_fd"]["CATEGORY"].value_counts().to_dict())

Tipos de la etiqueta: {'entrenamiento': 'str', 'prueba_dd': 'str', 'prueba_fd': 'bool'}
Valores v2: {True: 286, False: 286}


## 3. Partición de entrenamiento: balance, temas y longitud

In [5]:
tr = data.cargar_entrenamiento()
tr["clase"] = tr["etiqueta"].map({1: "Falsa", 0: "Verdadera"})
tr["n_palabras"] = tr["texto"].str.split().str.len()
print(tr["clase"].value_counts().to_string())
temas = pd.crosstab(tr["tema"], tr["clase"]).sort_values("Falsa", ascending=False)
temas["% falsas"] = (100 * temas["Falsa"] / temas.sum(axis=1)).round(1)
temas

clase
Falsa        338
Verdadera    338


clase,Falsa,Verdadera,% falsas
tema,,,
Politics,105,121,46.5
Entertainment,55,48,53.4
Society,52,41,55.9
Sport,41,45,47.7
Science,30,32,48.4
Security,18,11,62.1
Health,16,16,50.0
Economy,12,18,40.0
Education,9,6,60.0


In [6]:
fig, ax = plt.subplots(figsize=(8, 4.2))
t = temas[["Falsa", "Verdadera"]].sort_values("Falsa")
y = np.arange(len(t))
ax.barh(y - 0.2, t["Falsa"], height=0.38, color=NARANJA, label="Falsa")
ax.barh(y + 0.2, t["Verdadera"], height=0.38, color=AZUL, label="Verdadera")
ax.set_yticks(y, t.index); ax.set_xlabel("Número de noticias")
ax.set_title("Noticias por tema y clase · partición de entrenamiento (n = 676)")
ax.legend(frameon=False, loc="lower right")
guardar(fig, "eda_temas.png"); fig

<Figure size 1600x840 with 1 Axes>

In [7]:
print(tr.groupby("clase")["n_palabras"].describe().round(0))
fig, ax = plt.subplots(figsize=(8, 4))
bins = np.linspace(0, 1500, 41)
for c, col in [("Falsa", NARANJA), ("Verdadera", AZUL)]:
    ax.hist(tr.loc[tr.clase == c, "n_palabras"].clip(upper=1500), bins=bins, histtype="step", lw=2, color=col, label=c)
ax.set_xlabel("Longitud de la noticia (palabras; valores > 1.500 agrupados en 1.500)")
ax.set_ylabel("Número de noticias")
ax.set_title("Distribución de la longitud por clase · entrenamiento")
ax.legend(frameon=False)
guardar(fig, "eda_longitud.png"); fig

           count   mean    std   min    25%    50%    75%     max
clase                                                            
Falsa      338.0  297.0  154.0  28.0  208.0  270.0  382.0  1688.0
Verdadera  338.0  463.0  309.0  62.0  266.0  383.0  563.0  2091.0


<Figure size 1600x800 with 1 Axes>

**Lectura.** Las clases están balanceadas (H1). Las falsas son más cortas (mediana 269 vs 382 palabras,
H6): el modelo podría usar la longitud como atajo; se revisará el desempeño por tercil de longitud.

## 4. Fuentes (medios): concentración y riesgo de aprender el medio

In [8]:
fuentes = tr.groupby(["clase", "fuente"]).size().rename("n").reset_index().sort_values("n", ascending=False)
top_f = fuentes[fuentes.clase == "Falsa"].head(8).copy()
top_f["% de las falsas"] = (100 * top_f["n"] / (tr.clase == "Falsa").sum()).round(1)
top_v = fuentes[fuentes.clase == "Verdadera"].head(8).copy()
top_v["% de las verdaderas"] = (100 * top_v["n"] / (tr.clase == "Verdadera").sum()).round(1)
display(top_f); display(top_v)
cuatro = top_f.head(4)["n"].sum() / (tr.clase == "Falsa").sum()
print(f"Las 4 fuentes con más falsas concentran el {cuatro:.0%} de las falsas.")
print("Fuentes distintas en entrenamiento:", tr["fuente_norm"].nunique())
mixtas = tr.groupby("fuente_norm")["etiqueta"].agg(["min", "max"])
print("Fuentes con noticias de ambas clases:", int((mixtas["min"] != mixtas["max"]).sum()))

,clase,fuente,n,% de las falsas
14,Falsa,El Dizque,99,29.3
17,Falsa,El Ruinaversal,55,16.3
27,Falsa,Hay noticia,31,9.2
7,Falsa,Censura 0,25,7.4
34,Falsa,La voz popular,16,4.7
50,Falsa,Retroceso,12,3.6
39,Falsa,Modo Noticia,11,3.3
21,Falsa,El mundo today,6,1.8


,clase,fuente,n,% de las verdaderas
98,Verdadera,El país,45,13.3
99,Verdadera,Excelsior,35,10.4
93,Verdadera,El Universal,27,8.0
101,Verdadera,Forbes,21,6.2
122,Verdadera,Proceso,18,5.3
82,Verdadera,EL UNIVERSAL,16,4.7
103,Verdadera,HUFFPOST,16,4.7
75,Verdadera,BBC News,12,3.6


Las 4 fuentes con más falsas concentran el 62% de las falsas.
Fuentes distintas en entrenamiento: 120
Fuentes con noticias de ambas clases: 2


In [9]:
fig, ax = plt.subplots(figsize=(8, 4.2))
f = top_f.sort_values("n")
ax.barh(f["fuente"], f["n"], color=NARANJA, height=0.6)
for i, (n, p) in enumerate(zip(f["n"], f["% de las falsas"])):
    ax.text(n + 1, i, f"{n} ({p:.0f} %)", va="center", color=TEXTO_2, fontsize=9)
ax.set_xlabel("Noticias falsas en entrenamiento (cantidad)")
ax.set_title("Medios con más noticias falsas · entrenamiento\nvarios son sitios de sátira (p. ej. El Dizque, El Ruinaversal)")
ax.set_xlim(0, f["n"].max() * 1.2)
guardar(fig, "eda_fuentes_falsas.png"); fig

<Figure size 1600x840 with 1 Axes>

**Lectura (H7).** Pocas fuentes, en su mayoría **satíricas**, producen gran parte de las falsas. Un clasificador
puede aprender "estilo de El Dizque" en lugar de "falsedad". Por eso se usa validación cruzada **agrupada por
fuente** además de la estratificada.

## 5. Controles de integridad entre particiones (sin mirar etiquetas de prueba)

In [10]:
arm = {k: data.armonizar(d) for k, d in crudos.items()}
s_tr = set(arm["entrenamiento"]["fuente_norm"])
for k in ("prueba_dd", "prueba_fd"):
    s = set(arm[k]["fuente_norm"])
    print(f"{k}: {len(s & s_tr)} de {len(s)} fuentes ({len(s & s_tr)/len(s):.0%}) también están en entrenamiento")

prueba_dd: 43 de 71 fuentes (61%) también están en entrenamiento
prueba_fd: 28 de 195 fuentes (14%) también están en entrenamiento


In [11]:
import json
dedup = json.loads(config.ruta("results/tablas/deduplicacion.json").read_text(encoding="utf-8"))
umbral = dedup["umbral_similitud"]
cand = data.casi_duplicados(arm["entrenamiento"]["texto"], arm["prueba_dd"]["texto"], umbral)
print("DD: casi duplicados con entrenamiento:", int(cand.es_duplicado.sum()), "| idénticos:", int((cand.sim_max > 0.999).sum()))
ejemplos = cand[cand.es_duplicado].copy()
ejemplos["titular_dd"] = arm["prueba_dd"].loc[ejemplos.index, "titular"].str[:60]
ejemplos["titular_entrenamiento"] = arm["entrenamiento"].loc[ejemplos.idx_ref, "titular"].str[:60].values
display(ejemplos[["sim_max", "titular_dd", "titular_entrenamiento"]].round(3))
print("Reporte de deduplicación aplicado:", {k: dedup[k] for k in ("prueba_dd", "prueba_fd")})

DD: casi duplicados con entrenamiento: 5 | idénticos: 0


,sim_max,titular_dd,titular_entrenamiento
18,0.997,Mexicano anuncia que contraerá matrimonio con un iPhone X,Privatizarán el cerro del Tepeyac para construir un complejo
64,0.984,Crean máquina que puede pasar en un NUM % el dolor del parto,Miguel Ángel Mancera enfrentará a Floyd Mayweather en el Zóc
176,0.991,"Para todo sacan lo del populismo, ni siquiera saben de qué s",Gobierno de Alfredo del Mazo inició con récord… de delitos
224,0.991,ERUVIEL ÁVILA DEJA EL PRI CAPITALINO PARA UNIRSE A LA CAMPAÑ,Registra Meade su candidatura presidencial por el PRI
266,0.979,La copa de amor de Peña Nieto y Angélica Rivera,La copa de amor de Peña Nieto y Angélica Rivera


Reporte de deduplicación aplicado: {'prueba_dd': {'antes': 295, 'eliminados': 5, 'ids_eliminados': [19, 65, 177, 225, 267], 'despues': 290}, 'prueba_fd': {'antes': 572, 'eliminados': 4, 'ids_eliminados': [187, 345, 427, 433], 'despues': 568}}


**Lectura (H9–H11).** Hay textos de la partición oficial de desarrollo que ya están en entrenamiento, algunos
con **titulares distintos** (posible error de alineación del corpus). Se eliminan de las pruebas
(`data.preparar`) y una prueba automática (`tests/test_particiones.py`) comprueba que no quede ninguno.

## 6. Formato: diferencias entre versiones que crearían un cambio de distribución artificial

In [12]:
filas = []
for k, d in crudos.items():
    d = d.rename(columns=str.lower)
    txt = d["text"].astype(str)
    titu = d["headline"].fillna("").astype(str)
    filas.append({"partición": k,
                  "con *NUMBER* (%)": round(100 * txt.str.contains(r"\*NUMBER\*").mean(), 1),
                  "con dígitos (%)": round(100 * txt.str.contains(r"\d").mean(), 1),
                  "cuerpo empieza con el titular (%)": round(100 * pd.Series([c.strip().lower().startswith(t.strip().lower()[:30]) and t.strip() != ""
                                                                               for t, c in zip(titu, txt)]).mean(), 1),
                  "titular vacío (%)": round(100 * (titu.str.strip() == "").mean(), 1)})
formato = pd.DataFrame(filas); formato

,partición,con *NUMBER* (%),con dígitos (%),cuerpo empieza con el titular (%),titular vacío (%)
0,entrenamiento,76.2,5.2,96.9,0.0
1,prueba_dd,79.3,5.4,98.6,0.0
2,prueba_fd,0.0,92.1,0.9,12.6


In [13]:
def prop_mayus(s):
    letras = [c for c in s if c.isalpha()]
    return sum(c.isupper() for c in letras) / max(1, len(letras))
tr["titular_mayus"] = tr["titular"].map(prop_mayus) >= 0.6
tr["exclamaciones"] = tr["texto"].str.count("!")
tr["comillas"] = tr["texto"].str.count('["“”«»]')
estilo_cls = tr.groupby("clase")[["titular_mayus", "exclamaciones", "comillas"]].mean().round(3)
estilo_cls.columns = ["titular en mayúsculas (proporción)", "signos ! por noticia", "comillas por noticia"]
estilo_cls

,titular en mayúsculas (proporción),signos ! por noticia,comillas por noticia
clase,,,
Falsa,0.154,0.305,6.317
Verdadera,0.015,0.104,9.080


**Lectura (H3–H5).** La v1 reemplazó cifras por `*NUMBER*` y la v2 conserva dígitos; si no se corrige, el
modelo aprendería un símbolo que no existe en las noticias nuevas. En v1 el cuerpo repite el titular. Los
titulares en mayúsculas son casi exclusivos de las falsas de v1 (formato del sitio). Decisiones: normalizar
cifras/URLs igual en todo (`preprocessing.normalizar`), quitar el titular repetido y medir el efecto de las
mayúsculas en E‑08.

## 7. Tabla de hallazgos (se exporta a `results/tablas/eda_hallazgos.csv`)

In [14]:
mfal = tr.loc[tr.clase == "Falsa", "n_palabras"].median(); mver = tr.loc[tr.clase == "Verdadera", "n_palabras"].median()
hall = pd.DataFrame([
    ("H1", "Balance de clases (entrenamiento)", f"{(tr.etiqueta==1).sum()} falsas / {(tr.etiqueta==0).sum()} verdaderas", "F1-macro como métrica principal"),
    ("H2", "Titulares vacíos en v2", f"{int(faltantes.loc['headline','prueba_fd'])} de {len(crudos['prueba_fd'])}", "Cadena vacía + subgrupo 'sin titular'"),
    ("H3", "Cifras normalizadas solo en v1", f"*NUMBER* en {formato.iloc[0,1]} % (v1) vs {formato.iloc[2,1]} % (v2)", "Normalizar todo a NUM"),
    ("H4", "Cuerpo repite el titular (v1)", f"{formato.iloc[0,3]} % en entrenamiento vs {formato.iloc[2,3]} % en v2", "Quitar titular repetido"),
    ("H5", "Titular en mayúsculas", f"{estilo_cls.iloc[0,0]:.1%} falsas vs {estilo_cls.iloc[1,0]:.1%} verdaderas", "Minúsculas por defecto; E-08"),
    ("H6", "Longitud distinta por clase", f"mediana {mfal:.0f} vs {mver:.0f} palabras", "Análisis por tercil de longitud"),
    ("H7", "Falsas concentradas en sitios satíricos", f"4 fuentes = {cuatro:.0%} de las falsas", "CV agrupada por fuente"),
    ("H8", "Fuentes compartidas con prueba DD", f"{len(set(arm['prueba_dd'].fuente_norm) & s_tr)} de {arm['prueba_dd'].fuente_norm.nunique()}", "Seleccionar con CV agrupada"),
    ("H9", "Casi duplicados DD–entrenamiento", f"{dedup['prueba_dd']['eliminados']} textos", "Eliminados de DD"),
    ("H10", "Casi duplicados FD–v1", f"{dedup['prueba_fd']['eliminados']} textos", "Eliminados de FD"),
], columns=["id", "hallazgo", "cuantificación", "tratamiento"])
hall.to_csv(config.ruta("results/tablas/eda_hallazgos.csv"), index=False)
hall

,id,hallazgo,cuantificación,tratamiento
0,H1,Balance de clases (entrenamiento),338 falsas / 338 verdaderas,F1-macro como métrica principal
1,H2,Titulares vacíos en v2,72 de 572,Cadena vacía + subgrupo 'sin titular'
2,H3,Cifras normalizadas solo en v1,*NUMBER* en 76.2 % (v1) vs 0.0 % (v2),Normalizar todo a NUM
3,H4,Cuerpo repite el titular (v1),96.9 % en entrenamiento vs 0.9 % en v2,Quitar titular repetido
4,H5,Titular en mayúsculas,15.4% falsas vs 1.5% verdaderas,Minúsculas por defecto; E-08
5,H6,Longitud distinta por clase,mediana 270 vs 383 palabras,Análisis por tercil de longitud
6,H7,Falsas concentradas en sitios satíricos,4 fuentes = 62% de las falsas,CV agrupada por fuente
7,H8,Fuentes compartidas con prueba DD,43 de 71,Seleccionar con CV agrupada
8,H9,Casi duplicados DD–entrenamiento,5 textos,Eliminados de DD
9,H10,Casi duplicados FD–v1,4 textos,Eliminados de FD
